In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ase.io import Trajectory
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor
import seaborn as sns
import matplotlib.pyplot as plt
col_list = sns.color_palette("tab10", 100)

import tqdm

from gnome_aqueous_stability.data_utils import Data_Handler
from gnome_aqueous_stability.analysis_utils import (
    plot_periodic_table_with_values, get_col_dict_for_atoms, 
    Stable_Entries, Stability_Criteria, get_simplified_df, 
    atoms_from_db, Compound_HHI_scores, sys_in_MP_db
)

# pHs = np.linspace(-2, 16, 19) # Data specific
# Us = np.linspace(-2, 4, 31) # Data specific

In [4]:
# Define directory of data (can be left as None if the data folder is in 
# the package root):
dir_of_data = '../../data/'

dh = Data_Handler(
# Whether to apply solid filter:
    solid_filter = True, 
# Whether to use only GGA calculations (True), or include r2SCAN data via the MP-mixing scheme (False):
    gga_only = False,
# Path to data directory:
    path_to_data_directory = dir_of_data
    )

In [5]:
elemental_HHI = pd.read_csv("../../data/HHI_values.csv")
elemental_HHI

,Element,HHI_P,HHI_R
0,He,3200.0,3900.0
1,Li,2900.0,4200.0
2,Be,8000.0,4000.0
3,B,2900.0,2000.0
4,C,500.0,500.0
...,...,...,...
72,Pt,5500.0,9100.0
73,Au,1100.0,1000.0
74,Hg,5500.0,3100.0
75,Tl,6500.0,6500.0


In [6]:
compounds_df = dh.get_df()
compounds_df

,Composition,MaterialId,Reduced Formula,Elements,NSites,Volume,Density,Point Group,Space Group,Space Group Number,...,Data Directory,gga_only_pbx_save_id,mixed_pbx_save_id,Disorder Probability,average_HHI_P,average_HHI_P_excluding_OHCNPS,max_HHI_P,average_HHI_R,average_HHI_R_excluding_OHCNPS,max_HHI_R
0,Cs1S6Zr3,000006a8c4,Cs(ZrS2)3,"[S, Zr, Cs]",10,302.9202,3.2834,2/m,C2/m,12,...,gs://crystal-design/output/borg/improved_wy_2/...,GGA_000006a8c4,GGA_000006a8c4,0.016508,2040,4050,6000,1980,3450,6000
1,Al18Fe1Ho4Pd1Pt4,00006650ea,Ho4Al18FePdPt4,"[Al, Fe, Pd, Ho, Pt]",28,469.8946,7.3787,2/m,C2/m,12,...,gs://crystal-design/output/borg/round_3_wy_upd...,GGA_00006650ea,GGA_00006650ea,0.911403,3371,3371,9500,2721,2721,9100
2,Al2B24Lu8Tc2,00008f2ec4,Lu4AlTcB12,"[B, Al, Tc, Lu]",36,373.1385,8.4961,2/m,P2/m,10,...,gs://crystal-design/output/borg/round_3_4/1e_3...,GGA_00008f2ec4,GGA_00008f2ec4,0.999102,4376,4376,9500,2200,2200,3100
3,B24Na1Nd2Pr1,0000a1b980,NaPr(NdB12)2,"[B, Na, Pr, Nd]",28,284.6332,4.1529,mmm,Cmmm,65,...,gs://crystal-design/output/borg/round_3_wy_3/1...,GGA_0000a1b980,GGA_0000a1b980,0.779137,3543,3543,9500,2064,2064,3100
4,Nd7Os1Pr3Si5,00011f62be,Pr3Nd7Si5Os,"[Si, Pr, Nd, Os]",16,430.2851,6.8040,1,P1,1,...,gs://crystal-design/output/borg/round_3_wy_3/1...,GGA_00011f62be,r2S_00011f62be,0.998717,7750,7750,9500,2819,2819,9100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
554049,Ag2Al3Sr1,ffffa131f1,SrAl3Ag2,"[Al, Sr, Ag]",6,124.1020,5.1421,6/mmm,P6/mmm,191,...,gs://crystal-design/output/borg/improved_wy_ro...,GGA_ffffa131f1,Not computed,0.828687,1900,1900,4200,1467,1467,3000
554050,Li8Nd5Sb1Sn7Tb3,ffffae57de,Li8Tb3Nd5Sn7Sb,"[Li, Sn, Sb, Nd, Tb]",24,587.4456,6.2364,m,Cm,8,...,gs://crystal-design/output/borg/round_3_wy_upd...,GGA_ffffae57de,GGA_ffffae57de,0.990502,5221,5221,9500,3042,3042,4200
554051,Co4Er2Tb5Tm5,ffffce2124,Tb5Er2Tm5Co4,"[Co, Tb, Er, Tm]",16,394.7692,9.2941,1,P1,1,...,gs://crystal-design/output/borg/round_3_wy_3/1...,GGA_ffffce2124,GGA_ffffce2124,0.931525,7900,7900,9500,3000,3000,3100
554052,Au3Cu1Sc8Zn4,fffff156dc,Sc8Zn4CuAu3,"[Sc, Cu, Zn, Au]",16,305.1101,6.9431,4/mmm,P4/mmm,123,...,gs://crystal-design/output/borg/round_6_4/1e_3...,GGA_fffff156dc,GGA_fffff156dc,0.512748,3456,3456,5500,3006,3006,4500


In [7]:
from pymatgen.core import Composition

# Create a lookup table so each element maps directly to its HHI_P and HHI_R values.
hhi_lookup = elemental_HHI.set_index("Element")[["HHI_P", "HHI_R"]].to_dict("index")

# Compute the requested HHI summary statistics for a single composition string.
def compute_hhi_metrics(formula):
    # Convert the composition string into element counts, e.g. {'Fe': 2, 'O': 3}.
    element_amounts = Composition(formula).get_el_amt_dict()

    def filtered_elements(ignored_elements=None):
        # Remove elements that should be ignored before checking whether HHI data is missing.
        ignored_elements = set() if ignored_elements is None else set(ignored_elements)
        return {
            element: amount
            for element, amount in element_amounts.items()
            if element not in ignored_elements
        }

    def has_missing_hhi(filtered_amounts):
        # If any non-ignored element has no HHI entry, we mark the metric as missing.
        return any(element not in hhi_lookup for element in filtered_amounts)

    def weighted_average(hhi_key, ignored_elements=None):
        # Atom-count-weighted average HHI for the elements included in the formula.
        filtered_amounts = filtered_elements(ignored_elements)
        if has_missing_hhi(filtered_amounts):
            return np.nan
        total_atoms = sum(filtered_amounts.values())
        if total_atoms == 0:
            return np.nan
        weighted_sum = sum(
            hhi_lookup[element][hhi_key] * amount
            for element, amount in filtered_amounts.items()
        )
        return int(round(weighted_sum / total_atoms))

    def max_value(hhi_key, ignored_elements=None):
        # Largest elemental HHI value present anywhere in the compound.
        filtered_amounts = filtered_elements(ignored_elements)
        if has_missing_hhi(filtered_amounts):
            return np.nan
        if not filtered_amounts:
            return np.nan
        return int(max(hhi_lookup[element][hhi_key] for element in filtered_amounts))

    return {
        "average_HHI_P": weighted_average("HHI_P"),
        "average_HHI_R": weighted_average("HHI_R"),
        "average_HHI_P_excluding_O_H": weighted_average("HHI_P", ignored_elements=["O", "H"]),
        "average_HHI_R_excluding_O_H": weighted_average("HHI_R", ignored_elements=["O", "H"]),
        "max_HHI_P": max_value("HHI_P"),
        "max_HHI_R": max_value("HHI_R"),
    }

# Keep a small identifier block from compounds_df, then append the computed HHI columns.
compound_hhi_df = compounds_df[["MaterialId", "Composition", "Reduced Formula"]].copy()
compound_hhi_df = pd.concat(
    [
        compound_hhi_df,
        pd.DataFrame(
            # Apply the helper to every composition and expand the returned dictionaries into columns.
            compounds_df["Composition"].map(compute_hhi_metrics).tolist(),
            index=compounds_df.index,
        ),
    ],
    axis=1,
)
compound_hhi_df.drop(columns=["Composition", 'Reduced Formula'], inplace=True)


In [8]:
compound_hhi_df

,MaterialId,average_HHI_P,average_HHI_R,average_HHI_P_excluding_O_H,average_HHI_R_excluding_O_H,max_HHI_P,max_HHI_R
0,000006a8c4,2040.0,1980.0,2040.0,1980.0,6000.0,6000.0
1,00006650ea,3371.0,2721.0,3371.0,2721.0,9500.0,9100.0
2,00008f2ec4,NaN,NaN,NaN,NaN,NaN,NaN
3,0000a1b980,3543.0,2064.0,3543.0,2064.0,9500.0,3100.0
4,00011f62be,7750.0,2819.0,7750.0,2819.0,9500.0,9100.0
...,...,...,...,...,...,...,...
554049,ffffa131f1,1900.0,1467.0,1900.0,1467.0,4200.0,3000.0
554050,ffffae57de,5221.0,3042.0,5221.0,3042.0,9500.0,4200.0
554051,ffffce2124,7900.0,3000.0,7900.0,3000.0,9500.0,3100.0
554052,fffff156dc,3456.0,3006.0,3456.0,3006.0,5500.0,4500.0


In [9]:
compound_hhi_df.to_csv("../../data/precomputed_HHI_scores.csv", index=False)